# Cardioformer-CKD — End-to-end experiments (Thesis C)

Prognostic incident-CKD **risk prediction** from **MIMIC-IV + MIMIC-IV-ECG**, adapting the
Cardioformer ECG transformer. Run cells top to bottom. Requires credentialed
PhysioNet data (no data is shipped in this repo).


## 0. Setup


In [ ]:
# !pip install -r requirements.txt
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
MIMIC = '/path/to/mimiciv/3.1'        # contains hosp/
ECG   = '/path/to/mimic-iv-ecg/1.0'   # contains record_list.csv + files/
HOSP  = f'{MIMIC}/hosp'
OUT   = 'dataset/ckd'


## 1. Build the incident-CKD cohort (index times, exclusions, labels, censoring)

Default horizon = 2 years. Change `--horizon_days` to 365 (1y) or 1825 (5y).

In [ ]:
!python -m data_preprocessing.build_cohort \
  --patients   {HOSP}/patients.csv.gz \
  --admissions {HOSP}/admissions.csv.gz \
  --diagnoses  {HOSP}/diagnoses_icd.csv.gz \
  --labevents  {HOSP}/labevents.csv.gz \
  --ecg_record_list {ECG}/record_list.csv \
  --out_dir {OUT} --horizon_days 730 --blank_days 30 \
  --baseline_lookback_days 365 --n_intervals 8

## 2. Extract structured EHR features (labs / vitals / comorbidities before t0)

In [ ]:
!python -m data_preprocessing.extract_ehr_features \
  --cohort {OUT}/ckd_cohort_labels.parquet \
  --patients {HOSP}/patients.csv.gz --admissions {HOSP}/admissions.csv.gz \
  --diagnoses {HOSP}/diagnoses_icd.csv.gz --labevents {HOSP}/labevents.csv.gz \
  --out_dir {OUT}

## 3. Preprocess ECG waveforms (WFDB -> filtered, resampled, normalized .npy)

In [ ]:
!python -m data_preprocessing.extract_ecg \
  --cohort {OUT}/ckd_cohort_labels.parquet \
  --ecg_root {ECG} --out_dir {OUT}/ecg_npy --target_fs 250

## 4. Link modalities and make subject-independent splits

In [ ]:
!python -m data_preprocessing.link_ecg_ehr \
  --cohort {OUT}/ckd_cohort_labels.parquet --ehr {OUT}/ehr_features.parquet \
  --ecg_manifest {OUT}/ecg_npy/ecg_manifest.parquet --out_dir {OUT} --policy all
!python -m data_preprocessing.make_splits \
  --dataset {OUT}/ckd_dataset.parquet --out_dir {OUT} \
  --train_frac 0.6 --val_frac 0.2 --seed 41

## 5. Quick label-balance / cohort sanity check

In [ ]:
import pandas as pd
df = pd.read_parquet(f'{OUT}/ckd_dataset_split.parquet')
print(df.groupby('split')['label_binary'].agg(['count','mean']))
print('event rate (any):', df['event_indicator'].mean())

## 6. Train — multimodal, discrete-time survival head (default)

In [ ]:
!python run.py --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_survival_multimodal --is_training 1

## 7. Risk stratification (Thesis C)

`test()` fits risk cut-points on the **validation** split and applies them to test. Default: median split -> low / high risk. Re-stratify a trained model without retraining:

In [ ]:
!python run.py --is_training 0 --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_survival_multimodal --risk_strat_quantiles 0.33,0.67 \
  --results results/ckd_prognosis/tertiles
!python scripts/plot_results.py --run results/ckd_prognosis/ckd_2y_survival_multimodal
import json
print(json.dumps(json.load(open('results/ckd_prognosis/ckd_2y_survival_multimodal/risk_groups.json'))['groups'], indent=1))

## 8. Ablation study (Thesis C)

One-factor-at-a-time ablations over modality, fusion, prognostic layer, ECG backbone components, EHR feature groups and training, 3 seeds each. Plan: `configs/ablations.yaml`.

In [ ]:
!python run_ablations.py --plan configs/ablations.yaml --dry_run | head
!python run_ablations.py --plan configs/ablations.yaml
!python collect_results.py --plan configs/ablations.yaml
!python scripts/plot_results.py --ablation results/ckd_prognosis/ablation_summary.csv

## 9. Inspect per-patient risk

In [ ]:
import pandas as pd
pred = pd.read_csv('results/ckd_prognosis/ckd_2y_survival_multimodal/test_predictions.csv')
print(pred.groupby('risk_group')[['abs_risk_365d','abs_risk_730d','event_by_horizon']].mean())
pred.sort_values('risk_score', ascending=False).head()

## 10. Pipeline check without MIMIC (synthetic data -- results meaningless)

In [ ]:
!python scripts/make_synthetic_dataset.py --out_dir dataset/synthetic --n_subjects 400
!python run.py --config configs/ckd_prognosis.yaml --root_path dataset/synthetic \
  --setting synthetic_check --train_epochs 2 --use_gpu 0 --num_workers 0